In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
os.system("cp /content/drive/MyDrive/.secrets/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json")
sys.path.insert(0, '/content/drive/MyDrive/repos/deep-learning')
!pip install -q mlflow pyyaml ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
print("✅ Runtime Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Runtime Ready


In [ ]:
import os
target = '/content/drive/MyDrive/data_lake/01_bronze_medical/polypgen'
print(f"Exists: {os.path.exists(target)}")
if os.path.exists(target):
    print(os.listdir(target)[:5])
else:
    bronze = '/content/drive/MyDrive/data_lake/01_bronze_medical'
    dirs = [d for d in os.listdir(bronze) if 'polyp' in d.lower()]
    print(f"Potential dirs: {dirs}")

In [ ]:
print("Starting Data Ingestion for PolypGen...")
!python /content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py --root /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen --out /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json

Starting Data Ingestion for PolypGen...
🔄 Converting /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen to COCO format at /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json...
Wrote 0 images, 0 annotations


In [ ]:
print("Starting Data Ingestion for PolypGen...")
!python /content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py --root /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen --out /content/drive/MyDrive/data_lake/02_silver/polyp_coco

Starting Data Ingestion for PolypGen...
🔄 Converting /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen to COCO format at /content/drive/MyDrive/data_lake/02_silver/polyp_coco...
Traceback (most recent call last):
  File "/content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py", line 66, in <module>
    convert(root, out)
  File "/content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py", line 36, in convert
    with open(out,'w') as f: json.dump(coco,f)
         ^^^^^^^^^^^^^
IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/data_lake/02_silver/polyp_coco'


In [ ]:
!ls /content/drive/MyDrive/data_lake/02_silver/polyp_coco/ | head -n 20

In [ ]:
!find /content/drive/MyDrive/data_lake/ -name "*.jpg" | head -n 10

/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0002 (1).jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0002.jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0004 (1).jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0004 (2).jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0004.jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0006 (1).jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0006.jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0010.jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0012 (1).jpg
/content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/100H0012.jpg


In [ ]:
!python -c "import json; d=json.load(open('/content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json')); print([x['file_name'] for x in d['images'][10:15]])"


[]


In [ ]:
!grep -c "file_name" /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json

In [ ]:
!python eval.py!cat eval.py

python3: can't open file '/content/drive/MyDrive/repos/deep-learning/experiments/009_sam2_zeroshot_eval/eval.py!cat': [Errno 2] No such file or directory


In [ ]:
!ls /content/drive/MyDrive/data_lake/02_silver/polyp_coco/ | grep ".jpg" | head -n 5

In [ ]:
!cat /content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py


In [ ]:
!cat eval.py

import os, json, yaml, sys, cv2
import torch
import numpy as np
from ultralytics import YOLO
from PIL import Image
from tqdm import tqdm

print("Loading SAM-2...")
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    if not os.path.exists("sam2_hiera_large.pt"):
        os.system("wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt")
    sam2_model = build_sam2("sam2_hiera_l.yaml", "sam2_hiera_large.pt", device="cuda")
    predictor = SAM2ImagePredictor(sam2_model)
except Exception as e:
    print("SAM-2 load failed:", e)

# Note: The script is launched from the experiment directory
print("Loading config...")
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

yolo_ckpt = cfg['model']['yolo_checkpoint']
coco_json_path = cfg['data']['coco_json']

print(f"Loading YOLOv8 from {yolo_ckpt}")
# Patch ultralytics settings to avoid interactivity
os.environ["YOLO_VERBOSE"] = "False"
yolo = YOLO(y

In [ ]:
!python eval.py

Loading SAM-2...
Loading config...
Loading YOLOv8 from /content/drive/MyDrive/models/trained/polyp_detection/yolov8m_run/weights/best.pt
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
0it [00:00, ?it/s]
✅ Evaluation complete. Processed 0 images, generated 0 masks.


In [ ]:
!grep -c "file_name" /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json
!cat config.yaml

0
data:
  coco_json: /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json
  dataset: polypgen
  image_root: /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen
experiment:
  name: 009_sam2_zeroshot_eval
mlflow:
  experiment_name: 009_sam2_zeroshot_eval
model:
  sam2_checkpoint: sam2_hiera_large.pt
  yolo_checkpoint: /content/drive/MyDrive/models/trained/polyp_detection/yolov8m_run/weights/best.pt


In [ ]:
!mkdir -p /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen
!cp /content/drive/MyDrive/data_lake/06_inference/polyp_detection/custom_polyps/*.jpg /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen/
!python /content/drive/MyDrive/repos/data-ingestion/scripts/ingest_polypgen.py --root /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen --out /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json
!python eval.py

🔄 Converting /content/drive/MyDrive/data_lake/01_bronze_medical/polypgen to COCO format at /content/drive/MyDrive/data_lake/02_silver/polyp_coco/polypgen_annotations.json...
Wrote 0 images, 0 annotations
Loading SAM-2...
Loading config...
Loading YOLOv8 from /content/drive/MyDrive/models/trained/polyp_detection/yolov8m_run/weights/best.pt
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
0it [00:00, ?it/s]
✅ Evaluation complete. Processed 0 images, generated 0 masks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')